In [2]:
import pandas as pd
import numpy as np

# ===============================
# LOAD FILE
# ===============================

file_path = "D:/farukhnagar plant part reports/Machine Part Report Plant 17 feb.xlsx"   # change to your file
df = pd.read_excel(file_path)

# Clean column names
df.columns = df.columns.str.strip()

# ===============================
# BASIC CALCULATIONS
# ===============================

# Reject %
df["Reject_%"] = np.where(
    df["Prodn"] > 0,
    (df["Rej"] / df["Prodn"]) * 100,
    0
)

# Quality Yield
df["Yield_%"] = np.where(
    df["Prodn"] > 0,
    (df["Ok"] / df["Prodn"]) * 100,
    0
)

# Cycle deviation %
df["Cycle_Deviation_%"] = np.where(
    df["CycleTime Tgt"] > 0,
    ((df["CycleTime Act"] - df["CycleTime Tgt"]) / df["CycleTime Tgt"]) * 100,
    0
)

# Cavity utilization %
df["Cavity_Utilization_%"] = np.where(
    df["Planned Cavity"] > 0,
    (df["Actual Cavity"] / df["Planned Cavity"]) * 100,
    0
)

# Availability indicator
df["Total_Loss_Time"] = (
    df["Downtime"] +
    df["CO Time"] +
    df["Break Time"]
)

# ===============================
# SUMMARY VIEW
# ===============================

summary_cols = [
    "Part Number",
    "Ok",
    "Rej",
    "Prodn",
    "Reject_%",
    "Yield_%",
    "Cycle_Deviation_%",
    "Cavity_Utilization_%",
    "Total_Loss_Time",
    "PE"
]

summary_df = df[summary_cols]

# ===============================
# SAVE
# ===============================

summary_df.to_excel("Production_17Feb_Diagnostic.xlsx", index=False)

print("Production diagnostic completed.")

Production diagnostic completed.


In [3]:
import pandas as pd
import numpy as np

# ===============================
# LOAD FILE
# ===============================

file_path = "D:/farukhnagar plant part reports/Machine Part Report Plant 17 feb.xlsx"   # change path if needed
df = pd.read_excel(file_path)

# Clean column names
df.columns = df.columns.str.strip().str.lower()

# ===============================
# CHECK REQUIRED COLUMNS
# ===============================

required_cols = [
    "Part Number",
    "Prodn",
    "Run Time",
    "CycleTime Act",
    "Actual Cavity",
    "Pdt Time"
]

missing = [col for col in required_cols if col not in df.columns]

if missing:
    print("Missing columns:", missing)
    print("Check column names in file.")
else:

    print("\n===== STEP 1: CAVITY TEST =====")

    # Theoretical production assuming cavity = running cavities
    df["expected_prodn"] = (
        (df["Run Time"] / df["CycleTime Act"]) * df["Actual Cavity"]
    )

    df["prodn_diff"] = df["Prodn"] - df["expected_prodn"]

    print(df[["Part Number", "Prodn", "expected_prodn", "prodn_diff"]].head(10))

    avg_diff = df["prodn_diff"].abs().mean()
    print("\nAverage production difference:", avg_diff)

    if avg_diff < df["Prodn"].mean() * 0.1:
        print("👉 Actual cavity likely represents running cavities.")
    else:
        print("👉 Actual cavity likely represents tool capacity or something else.")

    print("\n===== STEP 2: RUN TIME TEST =====")

    corr_runtime = df[["Run Time", "Prodn"]].corr().iloc[0,1]
    print("Correlation between Run Time and Production:", corr_runtime)

    if corr_runtime > 0.7:
        print("👉 Run time likely represents actual machine operating time.")
    else:
        print("👉 Run time may not be pure operating time — needs clarification.")

    print("\n===== STEP 3: PDT TIME TEST =====")

    corr_pdt = df[["Pdt Time", "Run Time"]].corr().iloc[0,1]
    print("Correlation between PDT Time and Run Time:", corr_pdt)

    if corr_pdt < 0:
        print("👉 PDT likely represents planned downtime.")
    else:
        print("👉 PDT meaning unclear — may not directly reduce run time.")

# ===============================
# SAVE RESULTS
# ===============================

df.to_excel("production_definition_test_results.xlsx", index=False)

print("\nDiagnostic results saved.")

Missing columns: ['Part Number', 'Prodn', 'Run Time', 'CycleTime Act', 'Actual Cavity', 'Pdt Time']
Check column names in file.

Diagnostic results saved.
